# RTS fusion experiments E1-E6 (Kaggle driver)
Thin driver for the code in `Experiments/`. Upload that folder (including `splits/grouped_folds.json`) as a Kaggle dataset, and add the competition release as a second dataset.
**Edit only the first cell.** Every run appends one row to `OUT_DIR/results/results.csv`; the last cell prints mean ± std across folds.

Only `OUT_DIR` is saved as notebook output: per run `best.pth`, `config_resolved.yaml` and `stats.json` (the fold's normalisation, needed for inference), plus `results/` with the CSVs, per-epoch `history/` and `figures/` (training curves, train/val/test bars, predicted chips). The tile cache and TensorBoard logs go to `/kaggle/temp`, which Kaggle does not save.

`FOLDS_FILE` is the leak-free grouped split (`make_grouped_split.py`): overlapping chips never straddle train/val/test, and its `test_ids` are held out of every fold. After each run the best checkpoint is also scored on that local test set (`results_test.csv`), which is the number to compare with the leaderboard.

One run (experiment × fold × seed, 50 epochs, fp32) is about 85–90 min on a T4 (~103 s/epoch for E1), so 20 runs need about 3 sessions of 12 h. Split them with `EXPERIMENTS` / `FOLDS`, and list earlier sessions' `results.csv` in `PREVIOUS_RESULTS`.

**DARTS pretraining (`DARTS_PRETRAIN = True`).** Each experiment is first trained for `PRETRAIN_EPOCHS` on DARTS (Sentinel-2 + ArcticDEM chips in the release's layout, `DARTS_EDA/export_darts_kaggle.py`; add it as a third dataset). At every DARTS val epoch the GeoAI chips outside the local test set are scored too (`geoai_*` columns in `darts_pretrain/results/history/`, report only). The pretrain run keeps `best.pth` and `last.pth`. The checkpoint picked on a DARTS val fold (`splits/darts_grouped_folds.json`, grouped by ground footprint) then starts `FINETUNE_EPOCHS` of training on each GeoAI fold in `FOLDS`, scored exactly like the other runs (names `<exp>_darts_ft`). DARTS chips never enter GeoAI val or test. DARTS RGB is reflectance (0-10000), so in pretraining it is mapped onto GeoAI's RGB distribution before the Satlas /255 (`data.rgb_ref=geoai`).

A DARTS epoch is ~2.5x a GeoAI one (~1250 train chips), so 100 epochs is ~14-15 h at `INPUT_SCALE` 2.0: more than one session. Training stops cleanly after `SESSION_HOURS`. Then add this session's output as an input dataset, set `PREVIOUS_OUT` to its `outputs` folder and run again: unfinished runs resume from `last.pth`, finished ones are reused, and the fine-tuning follows.

In [ ]:
# ============================== constants: edit only this cell ==============================
DATA_ROOT   = "/kaggle/input/rts-challenge/competition_release"  # release: train/, metadata/, tools/
CODE_ROOT   = "/kaggle/input/rts-fusion-code/Experiments"        # this repo's Experiments/ folder
OUT_DIR     = "/kaggle/working/outputs"   # saved output: runs/<exp>/<fold_seed>/{best.pth, config_resolved.yaml, stats.json}
                                          #   + results/ (CSVs, per-epoch history, figures)
CACHE_DIR   = "/kaggle/temp/cache"        # NaN-filled tiles (~1.2 GB) + fold stats; /kaggle/temp is not saved as output
LOG_DIR     = "/kaggle/temp/tb"           # TensorBoard events (not saved; the same numbers are in results/history/)
FOLDS_FILE  = "splits/grouped_folds.json"   # grouped CV + held-out local test (make_grouped_split.py)

EXPERIMENTS = ["e1", "e4"]   # e5 = E4's backbone + semantic segmentation head (instances = connected components)
                             # e2_gate = E2 + band gate, e2_boundary = E2 + boundary head, e6 = E2 + both (ablation: e2, e2_gate, e2_boundary, e6)
FOLDS       = [1]               # grouped-CV folds 0-4, or "all": train on all 756 labelled chips (local test included)
                                # for the final submission model. "all" has no val, so best.pth is the LAST epoch:
                                # set EPOCHS from the CV plateau (~35-40) and pick the recipe with CV runs first
SEEDS       = [0]

EPOCHS      = 50
BATCH_SIZE  = 2               # 8 runs out of memory on a T4 at INPUT_SCALE 2.0; earlier 1.0 runs used 8
NUM_WORKERS = 4
AMP         = False           # fp16 autocast makes Swin-V2 produce NaN / skipped steps; keep fp32
PRETRAINED  = True            # Swin_V2_B_Weights.NAIP_RGB_SI_SATLAS (needs internet)

DROP_NDWI        = False      # E2 ablation: 7 channels
NDWI_TO_TERRAIN  = False      # E3/E4 ablation: NDWI as a 4th terrain channel
MASK_LOGITS_INIT_STD = 0.001  # None = torchvision default init (loss_mask ~9 at start); 0.001 = detectron2-style

# RPN anchors (px). E1/E2/E4 (FPN): exactly 5 sizes, one per level P2-P6. E3 (single C5 map): any number.
ANCHOR_SIZES  = [16, 32, 64, 128, 256]    # in UPSCALED px: [8..128] (RTS 3rd-99th pct at native size) x INPUT_SCALE 2.0.
                                          # Use [8, 16, 32, 64, 128] when INPUT_SCALE = 1.0
ASPECT_RATIOS = [0.5, 1.0, 2.0]           # height/width, applied at every size

# Model input (identical for every experiment)
INPUT_SCALE = 2.0             # chips upsampled by this inside the model, train and eval; 1.0 = no upscaling.
                              # Outputs are mapped back to chip size. Time/memory grow ~INPUT_SCALE**2
OPTICAL        = "swin_v2_b"  # optical-stream network (trainable params of the whole E4/E5 model in brackets):
                              #   "swin_v2_b" Satlas aerial (106M) | "swin_v2_t" ImageNet (46M) | "resnet50" (49M E5 / 64M E4)
                              #   | "resnet34" (38M) | "resnet18" (28M). ImageNet ones need SWIN_NORM = "zscore"
FREEZE_OPTICAL = 0            # optical stages kept fixed, 0-4 (4 = whole optical network frozen: swin_v2_b E5 -> 19M trainable)
SWIN_NORM   = "satlas"        # "satlas": Swin RGB / 255 as the Satlas weights expect (NIR etc. matched to it);
                              # "zscore": per-band z-score, the old behaviour. Terrain stream is z-scored either way

# Online hard example mining in the RoI heads (identical for every experiment)
OHEM        = False           # True: train the box/mask heads on the hardest RoIs instead of a random 512 per image
OHEM_ROIS   = 128             # hardest RoIs kept per image
OHEM_NMS    = 0.7             # NMS IoU among candidate RoIs before picking, so one hard region can't fill the batch; 1.0 = off

# Boundary head (e2_boundary, e6 only; ignored elsewhere)
BOUNDARY_WEIGHT = 1.0        # loss_boundary = this * (BCE + Dice) on each RoI's GT outline

# E5 segmentation head (ignored by e1-e4; e5 ignores ANCHOR_SIZES / ASPECT_RATIOS / OHEM)
SEG_THRESH      = 0.5         # RTS probability threshold before splitting into connected components
SEG_GAP_W0      = 10.0        # extra BCE weight on the background gap between two nearby RTS (U-Net); 0 = off
SEG_DICE_WEIGHT = 1.0         # loss = gap-weighted BCE + this * Dice

# Train-time augmentation (identical for every experiment)
SCALE_JITTER   = None         # e.g. [0.75, 1.5]: resize each chip by a random factor; None = off
COPY_PASTE_P   = 0.0          # e.g. 0.5: chance per chip of pasting RTS cut from other train chips
COPY_PASTE_MAX = 3            # pasted RTS per chip ~ U{1..this}
BAND_DROP_P    = 0.0          # e.g. 0.1: chance per chip of replacing one of NIR/NDVI/NDWI/DEM/relief by its train mean
                              # (robustness to the test sites' NIR/DEM shift); use the same value for e2, e2_gate, e2_boundary, e6
RUN_TAG        = ""           # appended to the run name (e.g. "_aug" -> runs/e4_aug/); "" overwrites earlier runs

# DARTS pretraining -> fine-tuning on GeoAI
DARTS_PRETRAIN      = True    # True: each experiment trains on DARTS first, then fine-tunes on GeoAI FOLDS
DARTS_ROOT          = "/kaggle/input/rts-darts/kaggle_darts"   # DARTS_EDA/kaggle_darts: train/, metadata/, tools/
DARTS_FOLDS_FILE    = "splits/darts_grouped_folds.json"        # DARTS-only split, grouped by ground footprint (500 m)
PRETRAIN_FOLD       = 1       # DARTS fold whose val picks the pretrain checkpoint (fold 0 is Novaya Zemlya only)
PRETRAIN_EPOCHS     = 100
PRETRAIN_EVAL_EVERY = 5       # DARTS val every N epochs (and at the last); every epoch would add ~1 h
FINETUNE_EPOCHS     = 50      # GeoAI epochs after pretraining (EPOCHS is used when DARTS_PRETRAIN = False)
SESSION_HOURS       = 11.0    # training stops cleanly after the epoch that passes this; Kaggle kills at 12 h
PREVIOUS_OUT        = None    # OUT_DIR of an earlier session, e.g. "/kaggle/input/<its output>/outputs":
                              # finished runs there are reused, unfinished ones resume from their last.pth

RUN_SMOKE_TEST   = True
EVAL_TRAIN       = True       # also score the best checkpoint on the fold's train chips (train vs val vs test plot)
SAMPLE_CHIPS     = 8          # chips drawn with GT + predictions per split in results/figures/; 0 = none
PREVIOUS_RESULTS = []         # results.csv files from earlier sessions, merged into the summary

In [ ]:
import os, subprocess, sys
!pip install -q torchgeo timm
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # CODE_ROOT is read-only
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less fragmentation with variable chip sizes
sys.path.insert(0, CODE_ROOT)

if any(e != "e3" for e in EXPERIMENTS):
    assert len(ANCHOR_SIZES) == 5, "FPN experiments need exactly 5 ANCHOR_SIZES (P2-P6)"
ANCHOR_OVERRIDES = [f"model.anchor_sizes={list(map(int, ANCHOR_SIZES))}",
                    f"model.aspect_ratios={list(map(float, ASPECT_RATIOS))}"]
assert INPUT_SCALE > 0 and SWIN_NORM in ("satlas", "zscore")
assert OPTICAL == "swin_v2_b" or SWIN_NORM == "zscore", f"{OPTICAL} is ImageNet-pretrained: set SWIN_NORM = 'zscore'"
assert 0 <= FREEZE_OPTICAL <= 4
BACKBONE_OVERRIDES = [f"model.optical={OPTICAL}", f"model.freeze_optical={int(FREEZE_OPTICAL)}"]   # not passed to the smoke test, which covers every backbone
INPUT_OVERRIDES = [f"model.input_scale={float(INPUT_SCALE)}", f"data.swin_norm={SWIN_NORM}",
                   f"model.ohem={str(OHEM).lower()}", f"model.ohem_rois={int(OHEM_ROIS)}", f"model.ohem_nms={float(OHEM_NMS)}",
                   f"model.seg_thresh={float(SEG_THRESH)}", f"model.seg_gap_w0={float(SEG_GAP_W0)}",
                   f"model.seg_dice_weight={float(SEG_DICE_WEIGHT)}"]

def overrides(exp):
    o = [f"data.root={DATA_ROOT}", f"data.cache_dir={CACHE_DIR}", f"out_dir={OUT_DIR}", f"log_dir={LOG_DIR}",
         f"data.folds_file={FOLDS_FILE}",
         f"train.epochs={EPOCHS}", f"train.batch_size={BATCH_SIZE}", f"data.num_workers={NUM_WORKERS}",
         f"train.amp={str(AMP).lower()}", f"model.pretrained={str(PRETRAINED).lower()}",
         f"model.mask_logits_init_std={'null' if MASK_LOGITS_INIT_STD is None else MASK_LOGITS_INIT_STD}",
         *ANCHOR_OVERRIDES, *INPUT_OVERRIDES, *BACKBONE_OVERRIDES,
         f"aug.scale_jitter={'null' if SCALE_JITTER is None else list(map(float, SCALE_JITTER))}",
         f"aug.copy_paste_p={COPY_PASTE_P}", f"aug.copy_paste_max={COPY_PASTE_MAX}",
         f"aug.band_drop_p={float(BAND_DROP_P)}", f"model.boundary_weight={float(BOUNDARY_WEIGHT)}"]
    if RUN_TAG:
        o.append(f"name={exp}{RUN_TAG}")
    if exp in ("e2", "e2_gate", "e2_boundary", "e6"):   # all E2-based
        o.append(f"flags.drop_ndwi={str(DROP_NDWI).lower()}")
    if exp in ("e3", "e4", "e5"):
        o.append(f"flags.ndwi_to_terrain={str(NDWI_TO_TERRAIN).lower()}")
    return o

def run(script, *args):
    """Run a script from CODE_ROOT and stream its output into the cell."""
    cmd = [sys.executable, f"{CODE_ROOT}/{script}", *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/kaggle/working")
    tail = []
    for line in p.stdout:
        print(line, end="")
        tail = (tail + [line])[-30:]
    if p.wait():
        raise RuntimeError(f"{script} exited with {p.returncode}. Last output: " + "".join(tail))

import shutil, time
from config import load_config, run_name
T0 = time.time()

def train_run(cfg_path, fold, seed, ov, out_dir, extra=()):
    """train.py for one run, reusing or resuming PREVIOUS_OUT's copy of it.
    Returns "reused" (finished in an earlier session), "done", or "stopped" (session budget hit)."""
    rel = f"runs/{run_name(load_config(cfg_path, ov))}/fold{fold}_seed{seed}"
    here = f"{out_dir}/{rel}"
    resume = []
    if PREVIOUS_OUT:
        prev = os.path.normpath(f"{PREVIOUS_OUT}/{os.path.relpath(out_dir, OUT_DIR)}/{rel}")
        if os.path.exists(f"{prev}/best.pth") and not os.path.exists(f"{prev}/INCOMPLETE"):
            shutil.copytree(prev, here, dirs_exist_ok=True)
            print(f"{rel}: finished in an earlier session, reused")
            return "reused"
        if os.path.exists(f"{prev}/last.pth"):
            resume = ["--resume", f"{prev}/last.pth"]
    left = SESSION_HOURS - (time.time() - T0) / 3600
    if left < 0.5:
        print(f"{rel}: no session time left")
        return "stopped"
    run("train.py", "--config", cfg_path, "--fold", fold, "--seed", seed, *resume, *extra,
        "--stop_after_h", f"{left:.2f}", "--override", *ov)
    return "stopped" if os.path.exists(f"{here}/INCOMPLETE") else "done"

In [ ]:
# Smoke test: random 8x333x417 batch through every experiment (train + eval mode).
if RUN_SMOKE_TEST:
    run("smoke_test.py", *(["--pretrained"] if PRETRAINED else []), "--override", f"data.root={DATA_ROOT}", *ANCHOR_OVERRIDES, *INPUT_OVERRIDES)

In [ ]:
import json
stopped = False
for exp in EXPERIMENTS:
    cfg_path = f"{CODE_ROOT}/configs/{exp}.yaml"
    ov, init = overrides(exp), []
    if DARTS_PRETRAIN:
        pt_out = f"{OUT_DIR}/darts_pretrain"           # own runs/ and results/: DARTS scores never mix with GeoAI's
        pt_ov = [*ov, f"data.root={DARTS_ROOT}", f"data.cache_dir={CACHE_DIR}/darts", f"data.folds_file={DARTS_FOLDS_FILE}",
                 f"out_dir={pt_out}", f"name={exp}{RUN_TAG}_darts_pt", f"train.epochs={PRETRAIN_EPOCHS}",
                 f"train.eval_every={PRETRAIN_EVAL_EVERY}", "data.rgb_ref=geoai", "train.keep_last=true",
                 # GeoAI chips (all but the local test) also scored at every DARTS val epoch: report only
                 "monitor=" + json.dumps({"root": DATA_ROOT, "cache_dir": CACHE_DIR, "folds_file": FOLDS_FILE})]
        if train_run(cfg_path, PRETRAIN_FOLD, SEEDS[0], pt_ov, pt_out) == "stopped":
            print("DARTS pretraining continues next session: set PREVIOUS_OUT to this session's outputs.")
            stopped = True
            break
        init = ["--init", f"{pt_out}/runs/{run_name(load_config(cfg_path, pt_ov))}/fold{PRETRAIN_FOLD}_seed{SEEDS[0]}/best.pth"]
        ov = [*ov, f"name={exp}{RUN_TAG}_darts_ft", f"train.epochs={FINETUNE_EPOCHS}"]
    for fold in FOLDS:
        for seed in SEEDS:
            state = train_run(cfg_path, fold, seed, ov, OUT_DIR, extra=init)
            if state == "stopped":
                print("Fine-tuning continues next session: set PREVIOUS_OUT to this session's outputs.")
                stopped = True
                break
            if state == "reused":                 # scored in the earlier session (see PREVIOUS_RESULTS)
                continue
            ev = lambda split, *extra: run("eval.py", "--config", cfg_path, "--fold", fold,
                                           "--seed", seed, "--split", split, "--samples", SAMPLE_CHIPS, *extra,
                                           "--override", *ov)
            cv = fold != "all"           # "all" trained on val and local test chips: only train can be scored
            if SAMPLE_CHIPS and cv:
                ev("val", "--no_csv")     # train.py already wrote the val row; this only draws the chips
            if "grouped" in FOLDS_FILE and cv:   # best checkpoint on the held-out local test chips
                ev("test")
            if EVAL_TRAIN:                # best checkpoint on this fold's train chips, unaugmented
                ev("train")               # (fold "all": printed and drawn, no CSV row)
            run("plots.py", "--out_dir", OUT_DIR)
        if stopped:
            break
    if stopped:
        break

In [ ]:
import pandas as pd
for name, split in [("results_train.csv", "train"), ("results.csv", "val"), ("results_test.csv", "test")]:
    res = f"{OUT_DIR}/results/{name}"
    frames = [pd.read_csv(p) for p in (os.path.join(os.path.dirname(q), name) for q in PREVIOUS_RESULTS)
              if os.path.exists(p)]
    if frames:
        if os.path.exists(res):
            frames.append(pd.read_csv(res))
        os.makedirs(os.path.dirname(res), exist_ok=True)
        pd.concat(frames).to_csv(res, index=False)
    if os.path.exists(res):
        print(f"===== {split} =====")
        run("eval.py", "--summary", "--split", split, "--out_dir", OUT_DIR)
        display(pd.read_csv(res))
run("plots.py", "--out_dir", OUT_DIR)
from IPython.display import Image
if os.path.exists(f"{OUT_DIR}/results/figures/summary.png"):
    display(Image(f"{OUT_DIR}/results/figures/summary.png"))

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}